# RoadSense India: real-model GPU benchmark
Select a **GPU hardware accelerator** in Colab runtime settings before running. CUDA numbers are not pre-filled. This experiment measures latency of pretrained YOLOv8n, SegFormer-B0 and optional OWLv2; it does not establish Indian-road accuracy or autonomous-driving safety.
Run cells in order. Download the evidence ZIP before the session ends. See `docs/COLAB.md` and `docs/RESULTS.md` in the repository for methodology and publication rules.

In [ ]:
from pathlib import Path
import subprocess
import sys
REPO = Path('/content/RoadSense-India')
URL = 'https://github.com/Harsh-ryUk/RoadSense-India.git'
if not REPO.exists():
    subprocess.run(['git', 'clone', URL, str(REPO)], check=True)
else:
    origin = subprocess.check_output(['git', '-C', str(REPO), 'remote', 'get-url', 'origin'], text=True).strip()
    if origin != URL:
        raise RuntimeError('Existing directory has a different remote; use a fresh runtime.')
    print('Using existing checkout; not pulling or overwriting local changes.')
subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'requirements-dev.txt')], check=True)
print('If Colab requests a kernel restart, restart now and continue with the next cell.')

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import torch
REPO = Path('/content/RoadSense-India')
os.chdir(REPO)
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable. Select a GPU runtime; do not report this as a GPU run.')
print('Torch:', torch.__version__, 'CUDA runtime:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run(['nvidia-smi'], check=True)

In [ ]:
test_env = dict(os.environ, HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1', OPENBLAS_NUM_THREADS='1')
subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=REPO, env=test_env, check=True)
print('Offline tests passed; real-model measurement is next.')

## Run all three profiles
Default: repeated letterboxed `bus.jpg`, 1280×720, rendering on, 20 warmup + 100 measured frames per repeat, three repeats. Each profile gets a separate process. Full profiles force synchronous OWLv2 every ten frames; `core256` disables OWL and is **not an identical-output speedup**.
For your own video, upload through the Files pane, set `SOURCE` to its absolute path and choose a different output directory. Each repeat needs at least 120 decodable frames. CUDA stage boundaries synchronize completed work; this adds overhead and prevents overlap. First run downloads model weights.

In [ ]:
SOURCE = 'sample'
OUTPUT_DIRECTORY = REPO / 'runs' / 'colab_sample'
if OUTPUT_DIRECTORY.exists() and any(OUTPUT_DIRECTORY.iterdir()):
    raise RuntimeError('Output directory already has data. Choose a new OUTPUT_DIRECTORY to preserve it.')
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
run_env = dict(os.environ, OPENBLAS_NUM_THREADS='1', TOKENIZERS_PARALLELISM='false')
run_env.pop('HF_HUB_OFFLINE', None)
run_env.pop('TRANSFORMERS_OFFLINE', None)
subprocess.run([sys.executable, 'scripts/benchmark_pipeline.py', '--device', 'cuda', '--profile', 'all', '--source', SOURCE, '--frames', '100', '--warmup', '20', '--repeats', '3', '--threads', '2', '--output', str(OUTPUT_DIRECTORY / 'result.json')], cwd=REPO, env=run_env, check=True)

In [ ]:
from IPython.display import Image, Markdown, display
subprocess.run([sys.executable, 'scripts/summarize_benchmark.py', '--directory', str(OUTPUT_DIRECTORY)], cwd=REPO, check=True)
display(Markdown((OUTPUT_DIRECTORY / 'README.md').read_text()))
display(Image(filename=str(OUTPUT_DIRECTORY / 'comparison.png')))
display(Image(filename=str(OUTPUT_DIRECTORY / 'core256.png')))

In [ ]:
import shutil
from datetime import datetime, timezone
from google.colab import files
for name, command in [('pip-freeze.txt', [sys.executable, '-m', 'pip', 'freeze']), ('nvidia-smi.txt', ['nvidia-smi']), ('commit.txt', ['git', 'rev-parse', 'HEAD']), ('git-status.txt', ['git', 'status', '--porcelain'])]:
    (OUTPUT_DIRECTORY / name).write_text(subprocess.check_output(command, cwd=REPO, text=True))
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
archive = shutil.make_archive(str(OUTPUT_DIRECTORY.parent / ('roadsense_' + OUTPUT_DIRECTORY.name + '_' + stamp)), 'zip', root_dir=OUTPUT_DIRECTORY)
print('Download and review all evidence before updating the public README. Keep CPU results separate.')
files.download(archive)